# 02 · Consumo e Indexación
Carga los PDFs descargados, los divide en *chunks*, genera embeddings con
**Google text-embedding-004** y los indexa en una base vectorial **FAISS**
que se persiste en Drive.

| Componente | Herramienta |
|---|---|
| Carga de PDFs | `PyMuPDFLoader` (LangChain) |
| Segmentación | `RecursiveCharacterTextSplitter` |
| Embeddings | `GoogleGenerativeAIEmbeddings` (`text-embedding-004`) |
| Base vectorial | `FAISS` (local, guardada en Drive) |


In [ ]:
# ── Instalar dependencias ──────────────────────────────────────────────────
!pip install -q \
    langchain==0.3.14 \
    langchain-community==0.3.14 \
    langchain-google-genai==2.0.8 \
    faiss-cpu==1.9.0 \
    pymupdf==1.25.2
print('✓ Dependencias instaladas')


In [ ]:
# ── Montar Drive y configurar rutas ───────────────────────────────────────
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, json

BASE_DIR   = '/content/drive/MyDrive/RAG_BioMed'
CORPUS_DIR = f'{BASE_DIR}/corpus'
INDEX_DIR  = f'{BASE_DIR}/faiss_index'
MANIFEST   = f'{BASE_DIR}/manifest.json'

os.makedirs(INDEX_DIR, exist_ok=True)

# API keys almacenadas en Colab Secrets (ícono 🔑 del panel izquierdo)
GEMINI_KEY = userdata.get('GEMINI_API_KEY')

# Hiperparámetros de segmentación
CHUNK_SIZE    = 1000
CHUNK_OVERLAP = 200
EMBED_MODEL   = 'models/text-embedding-004'

print('✓ Configuración lista')


In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document

def cargar_documentos(manifest_path: str, corpus_dir: str) -> list[Document]:
    """
    Carga todos los PDFs listados en el manifiesto y enriquece sus metadatos.

    Por cada página se agregan:
        doc_id, titulo, autores, anio, arxiv_id, filename
    Estos campos se usan luego para las citas en la respuesta generada.

    Args:
        manifest_path : Ruta al archivo manifest.json.
        corpus_dir    : Directorio con los PDFs.

    Returns:
        Lista plana de Documents (una por página de cada PDF).
    """
    with open(manifest_path, 'r', encoding='utf-8') as f:
        manifest = json.load(f)

    todos = []
    for entrada in manifest:
        filepath = os.path.join(corpus_dir, entrada['filename'])
        if not os.path.exists(filepath):
            print(f'  ⚠  No encontrado: {entrada["filename"]}')
            continue
        try:
            loader = PyMuPDFLoader(filepath)
            paginas = loader.load()
            for pag in paginas:
                pag.metadata.update({
                    'doc_id'  : entrada['doc_id'],
                    'titulo'  : entrada['titulo'],
                    'autores' : ', '.join(entrada['autores'][:3]),
                    'anio'    : entrada['anio'],
                    'arxiv_id': entrada['arxiv_id'],
                    'filename': entrada['filename'],
                })
            todos.extend(paginas)
            print(f'  ✓  {entrada["doc_id"]}  ({len(paginas)} págs.)  {entrada["titulo"][:55]}')
        except Exception as e:
            print(f'  ✗  {entrada["doc_id"]}: {e}')

    print(f'\nTotal páginas cargadas: {len(todos)}')
    return todos


In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

def segmentar(documentos: list[Document]) -> list[Document]:
    """
    Divide las páginas en chunks de tamaño fijo con solapamiento.

    Agrega `chunk_id` = '<doc_id>_chunk_<NNN>' a los metadatos de cada
    fragmento, necesario para la trazabilidad de citas.

    Args:
        documentos : Lista de Documents cargados por `cargar_documentos`.

    Returns:
        Lista de Documents segmentados con chunk_id en metadata.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=['\n\n', '\n', '. ', ' ', '']
    )
    chunks = splitter.split_documents(documentos)

    # Asignar chunk_id único por documento
    contador: dict[str, int] = {}
    for chunk in chunks:
        did = chunk.metadata.get('doc_id', 'unk')
        contador[did] = contador.get(did, 0) + 1
        chunk.metadata['chunk_id'] = f'{did}_chunk_{contador[did]:03d}'

    print(f'{len(documentos)} páginas → {len(chunks)} chunks')
    print(f'Chunk size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}')
    return chunks


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

def crear_indice(chunks: list[Document], api_key: str,
                 index_dir: str) -> FAISS:
    """
    Genera embeddings con Google text-embedding-004 y construye un índice FAISS.

    El índice se guarda en `index_dir` para ser reutilizado por los demás
    notebooks sin necesidad de reindexar.

    Args:
        chunks    : Lista de chunks segmentados.
        api_key   : Google Gemini API key.
        index_dir : Directorio donde persistir el índice FAISS.

    Returns:
        Instancia FAISS cargada en memoria.
    """
    embeddings = GoogleGenerativeAIEmbeddings(
        model=EMBED_MODEL,
        google_api_key=api_key
    )
    print(f'Generando embeddings para {len(chunks)} chunks...')
    vs = FAISS.from_documents(chunks, embeddings)
    vs.save_local(index_dir)
    print(f'✓ Índice FAISS guardado → {index_dir}')
    print(f'  Vectores almacenados : {vs.index.ntotal}')
    return vs


In [ ]:
# ── Pipeline completo de indexación ───────────────────────────────────────
print('=== PIPELINE DE INDEXACIÓN ===\n')
documentos = cargar_documentos(MANIFEST, CORPUS_DIR)
chunks     = segmentar(documentos)
vs         = crear_indice(chunks, GEMINI_KEY, INDEX_DIR)

# Guardar estadísticas
stats = {
    'total_documentos': len(set(c.metadata['doc_id'] for c in chunks)),
    'total_paginas'   : len(documentos),
    'total_chunks'    : len(chunks),
    'chunk_size'      : CHUNK_SIZE,
    'chunk_overlap'   : CHUNK_OVERLAP,
    'embed_model'     : EMBED_MODEL,
    'vectores'        : int(vs.index.ntotal),
}
with open(f'{BASE_DIR}/index_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print('\nResumen final:')
for k, v in stats.items():
    print(f'  {k}: {v}')


In [ ]:
# ── Prueba rápida de recuperación ─────────────────────────────────────────
query_test = 'deep learning protein structure prediction'
resultados = vs.similarity_search(query_test, k=3)
print(f'Query: "{query_test}"\n')
for i, doc in enumerate(resultados, 1):
    m = doc.metadata
    print(f'[{i}] {m["doc_id"]}  chunk={m["chunk_id"]}  p.{m.get("page",0)+1}')
    print(f'    {doc.page_content[:120]}...')
    print()
